## Step 1: Install Dependencies

In [ ]:
!pip install -q crewai sentence-transformers faiss-cpu pandas python-dotenv

## Step 2: Set Environment Variables

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

# print(os.getenv("GEMINI_API_KEY"))

True

In [4]:
from dotenv import load_dotenv
load_dotenv()

from crewai import LLM

llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.2
)

response = llm.call("Hello")
print(response)

Hello! How can I help you today?


## Step 3: Load CSV

In [6]:
import pandas as pd

CSV_PATH = "tickets.csv"

df = pd.read_csv(CSV_PATH)
df.head()


,Unnamed: 0,Body,Department,Priority,Tags
0,0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec..."
1,1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']"
2,2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati..."
3,3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo..."
4,4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb..."


## Step 4: Create Metadata-Aware Text

In [7]:
texts=[]

for _,row in df.iterrows():
    text=f'''
Department: {row['Department']}
Priority: {row['Priority']}
Tags: {row['Tags']}
Ticket: {row['Body']}
'''
    texts.append(text)

texts[:2]


["\nDepartment: Technical Support\nPriority: high\nTags: ['Account', 'Disruption', 'Outage', 'IT', 'Tech Support']\nTicket: Dear Customer Support Team,I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.Could you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?\n",
 "\nDepartment: Returns and Exchanges\nPriority: medium\nTags: ['Product', 'Feature', 'Tech Support']\nTicket: Dear Customer Support Team,I hope this message reaches you well. I am reaching out to request detailed information about the capabilities of your smart home integration products listed on your website. As a potential custome

## Step 5: Generate Embeddings

In [8]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True
)

embeddings.shape


c:\projects\RAG_matrix\aGENTIC_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2358.41it/s]


(29651, 384)

## Step 6: Create FAISS Index

In [9]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

index.ntotal


29651

## Step 7: Search Similar Tickets

In [14]:
query = "What are the issues faced by macOS users?"

query_embedding = embedder.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
)

scores, indices = index.search(query_embedding,5)

indices


array([[11141,  8290, 13769, 23380,  4932]])

## Step 8: View Retrieved Tickets

In [15]:
for idx in indices[0]:
    print(texts[idx])
    print("="*80)



Department: Technical Support
Priority: high
Tags: ['Performance', 'IT', 'Tech Support']
Ticket: Facing performance challenges with the SaaS project management tool on macOS. These issues might be related to resource-intensive applications such as Ableton Live and HyperX Fury. Already attempted restarting and reducing background applications.


Department: Product Support
Priority: low
Tags: ['Network', 'Performance', 'Outage', 'Disruption', 'IT', 'Tech Support']
Ticket: Users encounter sporadic connectivity problems with the SaaS platform on MacBook Pro systems, particularly when using WeChat and Discord. Even after optimizing performance, restricting background applications, and adjusting network settings, the issues continue due to network bandwidth limitations.


Department: Technical Support
Priority: high
Tags: ['Performance', 'IT', 'Tech Support']
Ticket: Facing performance challenges with the SaaS project management tool on macOS. It seems that resource-intensive applications 

## Step 9: Create CrewAI Agents

In [12]:
from crewai import Agent, LLM

llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.2
)

retriever = Agent(
    role="Retriever",
    goal="Retrieve relevant tickets",
    backstory="You are an expert at finding relevant support tickets from the knowledge base.",
    llm=llm,
    verbose=True
)

analyst = Agent(
    role="Support Analyst",
    goal="Answer user questions using retrieved tickets",
    backstory="You are an experienced support analyst who answers only from the retrieved evidence.",
    llm=llm,
    verbose=True
)

## Step 10: Run Agentic RAG

In [16]:
query = "What are the issues faced by macOS users?"

# retrieved contexts from Step 8
context = "\n".join([texts[idx] for idx in indices[0]])

prompt = f'''
Question:
{query}

Context:
{context}
'''

response = llm.call(prompt)

print(response)


Based on the provided tickets, macOS users are facing the following issues:

1.  **Performance Challenges with SaaS Tools:**
    *   Users experience performance problems with SaaS project management tools on macOS.
    *   These issues are often linked to resource-intensive applications running concurrently (e.g., Ableton Live, HyperX Fury).
    *   General system performance problems on macOS are also noted, potentially due to incompatible software updates or network congestion.

2.  **Sporadic Connectivity Problems:**
    *   MacBook Pro users encounter inconsistent connectivity with SaaS platforms.
    *   This is particularly noticeable when using other applications like WeChat and Discord.
    *   The underlying cause is often attributed to network bandwidth limitations.

3.  **Software Compatibility Issues after Updates:**
    *   Specific applications (e.g., Screen Recorder) face difficulties and compatibility problems following macOS updates (e.g., macOS Monterey).
